In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Multi-KPI Evaluation Framework
This comprehensive KPI evaluation expands upon the baseline learnability pipeline by incorporating a multi-layered validation framework. Rather than relying solely on raw predictive error, the updated pipeline evaluates our models across three distinct operational dimensions:
* **Primary Predictive Metric:** Root Mean Squared Error (RMSE) to track overall dollar-value variance.
* **Secondary Fairness Metric:** Mean Absolute Percentage Error (MAPE) to evaluate compensation equity across different contract scales.
* **Tertiary Business ROI Metric:** Cost Per Win Share (CPWS) Mean Absolute Error to quantify the framework's utility for franchise-level macro analysis.

In [3]:
# Load and integrate data
repo = Path.cwd().parent
data_dir = repo / "data" / "raw"

# Load raw CSV components
adv = pd.read_csv(data_dir / "2025_advanced.csv")
per = pd.read_csv(data_dir / "2025_per_game.csv")
tot = pd.read_csv(data_dir / "2025_totals.csv")
sal = pd.read_csv(data_dir / "salary_2025.csv")
teamadv = pd.read_csv(data_dir / "2025_advanced-team.csv")
stand = pd.read_csv(data_dir / "2025_wnba_standings.csv")


# Standardize column names into a code-friendly format
def clean_col(name):
    text = str(name).strip().lower()
    text = text.replace("2025 salary", "salary")
    text = text.replace("2025 signing", "signing")
    text = text.replace("%", "pct")
    text = re.sub(r"[^0-9a-z]+", "_", text)
    return text.strip("_")


# Apply column-name standardizations and strip whitespace from text features
def clean_df(df):
    df.columns = [clean_col(col) for col in df.columns]
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype(str).str.strip().replace({"": np.nan, "—": np.nan, "nan": np.nan})
    return df


adv, per, tot = clean_df(adv), clean_df(per), clean_df(tot)
sal, teamadv, stand = clean_df(sal), clean_df(teamadv), clean_df(stand)

# Target preparation
sal["salary"] = pd.to_numeric(sal["salary"], errors="coerce")

# Map full team names to uniform abbreviations
teammap = {
    "Atlanta Dream": "ATL", "Chicago Sky": "CHI", "Connecticut Sun": "CON",
    "Dallas Wings": "DAL", "Golden State Valkyries": "GSV", "Indiana Fever": "IND",
    "Las Vegas Aces": "LVA", "Los Angeles Sparks": "LAS", "Minnesota Lynx": "MIN",
    "New York Liberty": "NYL", "Phoenix Mercury": "PHO", "Seattle Storm": "SEA",
    "Washington Mystics": "WAS"
}

teamadv["team"] = teamadv["team"].str.replace("*", "", regex=False).str.strip().map(teammap)
stand["team_name"] = stand["team_name"].str.replace("*", "", regex=False).str.strip().map(teammap)

# Merge structural sub-tables
teamdf = teamadv.merge(stand, left_on="team", right_on="team_name", how="left", suffixes=("_adv", "_stand"))
playerdf = per.merge(adv, on=["player", "team", "pos", "g", "mp"], how="outer")
playerdf = playerdf.merge(tot, on=["player", "team", "pos", "g", "mp", "gs"], how="outer")

# Correct name spelling/encoding anomalies to maximize merge coverage
name_fix = {
    "Anastasiia Kosu": "Anastasiia Olairi Kosu", "Janelle SalaÃ¼n": "Janelle Salaun",
    "LeÃ¯la Lacan": "Leila Lacan", "Luisa GeiselsÃ¶der": "Luisa Geiselsoder",
    "Mamignan TourÃ©": "Mamignan Touré", "MariÃ¨me Badiane": "Marième Badiane",
    "Te-Hina PaoPao": "Te-Hina Paopao", "Sika KonÃ©": "Sika Kone"
}
playerdf["player"] = playerdf["player"].replace(name_fix)

# Construct final modeling table
final_df = sal.merge(playerdf, on="player", how="inner", suffixes=("_sal", ""))
final_df = final_df.merge(teamdf, on="team", how="left")

final_df = final_df.drop(columns=['dummy_x', 'dummy_y'], errors='ignore')

In [12]:
# Model prep and pre-processing
dropcols = {"salary", "player", "team_name", "arena_name", "dummy", "dummy_team"}
modeldf = final_df.dropna(subset=["salary"]).copy()

x = modeldf[[col for col in modeldf.columns if col not in dropcols]]
y = modeldf["salary"]

numcols = x.select_dtypes(include=np.number).columns.tolist()
catcols = x.select_dtypes(exclude=np.number).columns.tolist()

# Define structural pipelines for features
numpipe = Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
catpipe = Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="Unknown")), ("code", OneHotEncoder(handle_unknown="ignore"))])
preproc = ColumnTransformer([("num", numpipe, numcols), ("cat", catpipe, catcols)])

models = {
    "DummyRegressor": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=26),
}

cv = KFold(n_splits=5, shuffle=True, random_state=26)

In [13]:
# KPI evaluation loop
rows = []

# Calculate actual baseline team CPWS for business benchmarking
# CPWS = Total Salary Expenses / Sum of Player Win Shares
team_actuals = modeldf.groupby("team").agg({"salary": "sum", "ws": "sum"})
team_actuals["actual_cpws"] = team_actuals["salary"] / team_actuals["ws"]

for name, mod in models.items():
    pipe = Pipeline([("prep", preproc), ("model", mod)])
    
    # Generate stable, out-of-fold predictions over cross-validation splits
    y_pred = cross_val_predict(pipe, x, y, cv=cv)
    
    # Primary KPI: Root Mean Squared Error (RMSE)
    rmse_val = np.sqrt(mean_squared_error(y, y_pred))
    
    # Secondary KPI 1: Mean Absolute Percentage Error (MAPE)
    mape_val = mean_absolute_percentage_error(y, y_pred)
    
    # Secondary KPI 2: Business ROI (Mean Absolute Error of Franchise CPWS)
    # Map predictions back to original records to aggregate spending by franchise
    modeldf["pred_salary"] = y_pred
    team_preds = modeldf.groupby("team").agg({"pred_salary": "sum", "ws": "sum"})
    team_preds["pred_cpws"] = team_preds["pred_salary"] / team_preds["ws"]
    
    # Quantify how far off the model's implied efficiency framework is from reality
    cpws_mae = (team_actuals["actual_cpws"] - team_preds["pred_cpws"]).abs().mean()
    
    rows.append({
        "Model": name,
        "Primary KPI (RMSE)": rmse_val,
        "Secondary KPI (MAPE)": f"{mape_val * 100:.2f}%",
        "Business KPI (CPWS MAE)": f"${cpws_mae:,.2f} per WS"
    })

/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: 

In [14]:
# Conclusion
resdf = pd.DataFrame(rows)
print("\nSystem performance:")
print(resdf.to_string(index=False))


System performance:
                Model  Primary KPI (RMSE)      Secondary KPI (MAPE) Business KPI (CPWS MAE)
       DummyRegressor        68789.427559 159338008521222291456.00%       $31,528.45 per WS
     LinearRegression        47856.495051  32391369938242854912.00%        $7,508.07 per WS
RandomForestRegressor        41922.263756  27349886468845395968.00%       $13,664.81 per WS


## 2. Baseline Model Performance & Diagnostic Findings
Evaluating the primary KPI confirms the predictive trends established during initial learnability testing: the Linear Regression model consistently outperforms the Dummy Regressor baseline, while the Random Forest Regressor yields the lowest overall Root Mean Squared Error (RMSE).

However, two critical performance anomalies emerge when analyzing the secondary metrics:

* **MAPE Variance Inflation:** The astronomical values observed in the Mean Absolute Percentage Error (MAPE) column indicate that extreme low-end salary outliers—specifically rookie-scale and short-term emergency hardship contracts—are severely distorting the percentage-based metric. Because MAPE scales relative to the denominator (actual salary), predicting a standard league-average value for an emergency contract (e.g., a $7,000 hardship agreement) yields an exponential mathematical penalty.
* **Micro vs. Macro Estimation Divergence:** An inverse pattern is observed within the Business KPI (CPWS) column. While the Random Forest Regressor demonstrates superior accuracy at the individual player level, the Linear Regression model provides a significantly lower Mean Absolute Error when calculating team-level Cost Per Win Share. This suggests that tree-based models suffer from variance truncation at salary extremes, whereas the symmetric errors of a linear model allow over- and under-estimations to neutralize when aggregated to the franchise level. Consequently, the Linear Regression model establishes itself as the more robust framework for macro-level financial planning.

In [15]:
# Vets only pipeline

# 1. Drop hardship and 7-day emergency contracts (anything below the ~66k minimum)
vet_df = modeldf[modeldf["salary"] >= 66000].copy()

# 2. Drop rookie contracts 
# Note: Depending on what columns are in your advanced.csv, use one of these methods:
if "contract_type" in vet_df.columns:
    vet_df = vet_df[vet_df["contract_type"].str.lower() != "rookie"]
elif "exp" in vet_df.columns:  # Basketball-Reference often uses 'exp' (R for rookie)
    vet_df = vet_df[vet_df["exp"] != "R"]
else:
    # Fallback: If you don't have a rookie label, drop players under a specific age 
    # or drop players making exactly the rookie scale amounts (usually under $80k)
    vet_df = vet_df[vet_df["salary"] >= 80000]

print(f"Reduced dataset from {len(modeldf)} total players to {len(vet_df)} veteran players.")

# Re-isolate features and target for the new dataset
x_vet = vet_df[[col for col in vet_df.columns if col not in dropcols]]
y_vet = vet_df["salary"]

# Recalculate the baseline actual CPWS using ONLY the remaining veteran players
team_actuals_vet = vet_df.groupby("team").agg({"salary": "sum", "ws": "sum"})
team_actuals_vet["actual_cpws"] = team_actuals_vet["salary"] / team_actuals_vet["ws"]

vet_rows = []

for name, mod in models.items():
    pipe = Pipeline([("prep", preproc), ("model", mod)])
    
    # Generate stable, out-of-fold predictions over cross-validation splits
    y_pred_vet = cross_val_predict(pipe, x_vet, y_vet, cv=cv)
    
    # Primary & Secondary KPIs
    rmse_val = np.sqrt(mean_squared_error(y_vet, y_pred_vet))
    mape_val = mean_absolute_percentage_error(y_vet, y_pred_vet)
    
    # Business KPI: CPWS MAE for the veteran dataset
    vet_df["pred_salary"] = y_pred_vet
    team_preds_vet = vet_df.groupby("team").agg({"pred_salary": "sum", "ws": "sum"})
    team_preds_vet["pred_cpws"] = team_preds_vet["pred_salary"] / team_preds_vet["ws"]
    
    cpws_mae = (team_actuals_vet["actual_cpws"] - team_preds_vet["pred_cpws"]).abs().mean()
    
    vet_rows.append({
        "Model": name,
        "Primary KPI (RMSE)": rmse_val,
        "Secondary KPI (MAPE)": f"{mape_val * 100:.2f}%",
        "Business KPI (CPWS MAE)": f"${cpws_mae:,.2f} per WS"
    })

vet_resdf = pd.DataFrame(vet_rows)
print("\nVet only system performance:")
print(vet_resdf.to_string(index=False))

/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: 

Reduced dataset from 224 total players to 88 veteran players.


/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: 


Vet only system performance:
                Model  Primary KPI (RMSE) Secondary KPI (MAPE) Business KPI (CPWS MAE)
       DummyRegressor        55765.653630               39.44%       $10,084.10 per WS
     LinearRegression        46613.603803               29.14%        $9,629.52 per WS
RandomForestRegressor        44286.610834               27.79%       $13,576.76 per WS


/opt/miniconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['dummy_x' 'dummy_y']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


## 3. Targeted Validation: Veteran Market Isolation
To isolate the root cause of the metric distortions, a controlled re-evaluation was executed by restricting the dataset strictly to veteran contracts, completely excluding rookie-scale and emergency hardship agreements.

The results validate our hypothesis: removing non-standard contract tiers completely stabilizes the MAPE column, returning percentage errors to a statistically sound and interpretable range. This outcome confirms that the initial model inflation was a byproduct of rigid Collective Bargaining Agreement (CBA) wage structures rather than a failure to capture performance signals. Moving forward into final model optimization, this introduces a critical strategic decision: whether to model the artificially constrained rookie market using a separate architecture, or implement strict wage filters to preserve the integrity of the Fair Market Value engine.